# `agent_free_chat` -- prototipo del agente que falta

`lora_brain` tiene tres agentes (`agent_router`, `agent_corrector`,
`agent_behavior`) y **ninguno de free chat**: hoy free chat son dos funciones
sueltas dentro de `orchestrator_node.py` (`responder()` y `_manejar_chat_libre()`).
El proyecto original tampoco lo tiene como agente.

Este notebook prototipa ese cuarto agente con la arquitectura V1:
**LLM + memoria episodica + BM25**. Sin embeddings, sin ChromaDB, sin reranker,
sin semantic chunking.

```
   Mensaje del usuario
            |
            v
     memoria de LO QUE CONTO EL USUARIO
            |
            v
       JSONL + BM25
            ^
            |
 Pregunta --+--> buscar --> threshold
                             /     \
                           SI       NO        <-- NUEVO: hoy no decide nada
                           /          \
                memoria->LLM      conversar y repreguntar
```

## Decision de diseno: la memoria guarda SOLO lo que dijo el usuario

Esto es lo que separa a este agente de lo que corre hoy, y es deliberado.
`memoria_episodica.py` indexa el turno completo (`usuario` + `respuesta`) e
inyecta **la respuesta vieja de Lora**:

```python
corpus_tok = [_tokenizar(f"{ep['usuario']} {ep['respuesta']}") ...]   # indexa las dos
respuesta = episodios[mejor_idx]["respuesta"]
return f"(Ya hablamos de esto antes: {respuesta})"                     # inyecta la de Lora
```

El problema: **lo que dijo Lora no es un hecho sobre el usuario**, es texto
generado por un 0.5B que el propio README documenta como poco confiable (falla
en aritmetica, a veces se desentiende de temas emocionales). Indexarlo hace que
el modelo se pueda enganchar con sus propios inventos y que le vuelvan como
"contexto confiable". Es la version *retrieval* del aviso que ya esta escrito en
el proyecto sobre el dataset de fine-tuning:

> nunca se entrena directamente con este archivo, para que el modelo no refuerce
> sus propios errores

Aca entonces: **se indexa y se inyecta solo el mensaje del usuario.**

| | `memoria_episodica.py` (hoy) | Este agente |
|---|---|---|
| Que se indexa | `usuario` + `respuesta` de Lora | solo `usuario` |
| Que se inyecta | la respuesta vieja de Lora | lo que conto el usuario |
| Texto del recuerdo | `(Ya hablamos de esto antes: ...)` | `(Antes me contaste: ...)` |

**Consecuencia medible, y es esperada:** van a enganchar menos recuerdos, porque
hay menos tokens por episodio. La seccion 4 lo mide. Es el precio correcto:
preferible pocos recuerdos ciertos que muchos que son eco del propio modelo.

## Sin Memory Extractor

Se probo resumir cada turno con el LLM antes de indexarlo. **Medido: empeora.**
El resumen pasaba 0/7 consultas contra 1/7 del turno crudo, porque:

1. un resumen tiene **menos** tokens, y BM25 necesita vocabulario para matchear;
2. el 0.5B **pierde datos** al resumir (`"mi hermano vive en Arequipa"` ->
   `"El usuario vive en Arequipa"`, se borro el hermano);
3. mete acentos que el usuario no escribio (`ingenieria` -> `ingeniería`), y el
   tokenizador los trata como palabras distintas.

Resumir sirve cuando se comprime MUCHO texto en poco (un transcript de 50 turnos
-> una frase). Un episodio de este robot ya es un turno corto: no hay nada que
comprimir, solo palabras que perder. Asi que el agente **no llama al LLM para
memorizar** -- cero costo extra por turno, coherente con `agent_router` y
`agent_corrector`, que tampoco usan LLM a proposito.

## Lo que si aporta el agente

El **threshold con tres ramas**, que hoy no existe: `buscar_relevante()` devuelve
`None` y nadie decide nada. Ver seccion 5.

## 1. Entorno

Se importan los modulos reales de `lora_brain` sin ROS 2 ni `colcon build`.
La memoria del agente va a un archivo **aparte**: `registro_chat.py` sigue
guardando el turno completo (lo necesita `curar.py` para armar datasets de
fine-tuning), y este archivo guarda solo los mensajes del usuario.

In [ ]:
import json
import os
import random
import sys
import time
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "Jupyter-notebook" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src" / "lora_brain"))

os.environ.setdefault("CHAT_SERVER_HOST", "http://localhost:11434")
os.environ.setdefault("CHAT_MODEL", "lora-chat-libre-v4")

from lora_brain import llama_client, memoria_episodica  # noqa: E402

# Memoria del agente: SOLO mensajes del usuario. Archivo propio, no se mezcla
# con el de registro_chat.py (que guarda el turno completo a proposito).
MIS_EPISODIOS = Path.home() / ".lora" / "chat_libre_training" / "mis_episodios.jsonl"
MIS_EPISODIOS.parent.mkdir(parents=True, exist_ok=True)

print("host      :", llama_client.CHAT_SERVER_HOST)
print("CHAT_MODEL:", llama_client.CHAT_MODEL)
print("memoria   :", MIS_EPISODIOS)

## 2. Lo que el agente ya sabe sin memoria

Parte del contexto no se recupera: **ya esta**. Dos fuentes, y conviene no
confundirlas con la memoria episodica:

- **Identidad del robot** -- horneada en el fine-tuning, no en un system prompt.
  El `Modelfile` del proyecto original dice *"SIN linea SYSTEM, a proposito"*, y
  el dataset trae los ejemplos (`"quien sos" -> "Soy Lora, tu robot"`). El agente
  **no** inyecta esto: el modelo ya lo sabe.
- **Nombre del usuario** -- lo pide el orchestrator al abrir sesion y vive en
  `estado["nombre"]`. Es dato de sesion, no un episodio.

In [ ]:
# Equivale a estado["nombre"] del orchestrator.
PERFIL = {"nombre": "Adrian"}

# Identidad del robot: NO se inyecta, esta en el fine-tune. Se lista para dejar
# explicito que es info que el agente ya tiene y no debe buscar ni repreguntar.
IDENTIDAD_HORNEADA = {
    "nombre": "Lora",
    "que_es": "un robot que charla, juega trivia y el juego de emociones",
}

print("perfil de sesion:", PERFIL)
print("identidad (fine-tune, no se inyecta):", IDENTIDAD_HORNEADA)
print()

# Se verifica que el modelo de verdad traiga la identidad horneada: si contesta
# cualquier cosa, el agente no puede confiar en que ya la sepa.
for pregunta in ("quien sos", "como te llamas"):
    r = llama_client.generar_respuesta(
        [{"role": "user", "content": pregunta}], max_tokens=30).strip()
    print(f"  {pregunta!r:<18} -> {r}")

## 3. La memoria: solo mis mensajes

Un episodio es **un mensaje del usuario**, y nada mas. No se guarda lo que
contesto Lora.

```json
{"ts": "...", "usuario": "mi hermano vive en Arequipa", "importancia": 0.9}
```

`importancia` se calcula con reglas, sin LLM: sirve para desempatar y para decidir
que podar cuando el archivo crezca.

Se filtran los turnos que no cuentan nada (saludos, preguntas del usuario): no son
episodios y solo ensucian el indice BM25.

BM25 se arma con el **mismo tokenizador** de `memoria_episodica.py`, para que la
comparacion de la seccion 4 sea justa: lo unico que cambia es QUE texto se indexa.

In [ ]:
from rank_bm25 import BM25Okapi

_tokenizar = memoria_episodica._tokenizar   # mismo tokenizador que el modulo real

# Turnos que no son episodios: no cuentan nada de la vida del usuario.
_TRIVIALES = {
    "hola", "buenas", "chau", "adios", "gracias", "ok", "dale", "si", "no",
    "bien", "nada", "listo", "bueno", "perfecto", "jaja",
}
_PALABRAS_MINIMAS = 3
_INTERROGATIVOS = (
    "que ", "qué ", "quien ", "quién ", "como ", "cómo ", "cual ", "cuál ",
    "donde ", "dónde ", "cuando ", "cuándo ", "por que ", "por qué ",
)
# Apelaciones a la memoria: "te acordas de mi trabajo?" PREGUNTA por un dato,
# no lo aporta. Sin esto se guardaba como episodio y contaminaba el indice con
# la pregunta en lugar de la respuesta -- visto corriendo la seccion 7.
_PIDE_RECUERDO = (
    "te acordas", "te acuerdas", "recordas", "recuerdas", "te conte",
    "te conté", "te dije", "hablamos de", "sabes cual",
)


def vale_la_pena(mensaje_usuario):
    """False si el mensaje no cuenta nada que valga recordar.

    Descarta saludos, mensajes de menos de 3 palabras, preguntas del usuario y
    apelaciones a la memoria: un episodio tiene que ser algo que el usuario
    CONTO, no algo que pidio.
    """
    limpio = mensaje_usuario.strip().lower().strip("¿?¡!.,")
    if limpio in _TRIVIALES or len(limpio.split()) < _PALABRAS_MINIMAS:
        return False
    if limpio.startswith(_INTERROGATIVOS):
        return False
    if any(p in limpio for p in _PIDE_RECUERDO):
        return False
    return True


def _importancia(mensaje_usuario):
    """Prioridad por reglas, sin LLM. Identidad > preferencia > comentario."""
    t = mensaje_usuario.lower()
    if any(p in t for p in ("me llamo", "mi nombre", "tengo ", "vivo en",
                            "trabajo", "estudio", "mi hermano", "mi mama",
                            "mi papa", "mi perro", "mi gato")):
        return 0.9
    if any(p in t for p in ("me gusta", "odio", "prefiero", "quiero",
                            "me encanta", "sueño")):
        return 0.7
    return 0.5


def guardar(mensaje_usuario, ruta=None):
    """Guarda el mensaje del usuario como episodio. Devuelve el episodio o None.

    Silencioso ante error de disco, igual criterio que registro_chat.registrar():
    perder una linea del log nunca puede cortar la conversacion.
    """
    if not vale_la_pena(mensaje_usuario):
        return None
    ep = {
        "ts": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "usuario": mensaje_usuario.strip(),
        "importancia": _importancia(mensaje_usuario),
    }
    ruta = Path(ruta or MIS_EPISODIOS)
    try:
        ruta.parent.mkdir(parents=True, exist_ok=True)
        with open(ruta, "a", encoding="utf-8") as f:
            f.write(json.dumps(ep, ensure_ascii=False) + "\n")
    except OSError as e:
        print(f"[memoria] no se pudo guardar ({e})")
    return ep


def cargar(ruta=None):
    ruta = Path(ruta or MIS_EPISODIOS)
    if not ruta.exists():
        return []
    eps = []
    for linea in ruta.read_text(encoding="utf-8").splitlines():
        if linea.strip():
            try:
                eps.append(json.loads(linea))
            except ValueError:
                pass
    return eps


def buscar(mensaje_usuario, episodios=None):
    """(episodio, overlap, score) del mejor candidato, o (None, 0, 0.0).

    Indexa SOLO el mensaje del usuario -- nunca lo que contesto Lora.
    Mismo criterio de ranking que memoria_episodica.buscar_relevante(): manda
    el overlap de palabras de contenido y el score BM25 solo desempata.
    NO aplica threshold: eso lo decide decidir().
    """
    episodios = cargar() if episodios is None else episodios
    query_tok = set(_tokenizar(mensaje_usuario))
    if not query_tok or not episodios:
        return None, 0, 0.0

    corpus_tok = [_tokenizar(e["usuario"]) for e in episodios]
    if not any(corpus_tok):
        return None, 0, 0.0
    scores = BM25Okapi(corpus_tok).get_scores(list(query_tok))

    mejor_i, mejor_ov, mejor_sc = None, 0, 0.0
    for i, tokens in enumerate(corpus_tok):
        ov = len(query_tok & set(tokens))
        if ov > mejor_ov or (ov == mejor_ov and ov > 0 and scores[i] > mejor_sc):
            mejor_i, mejor_ov, mejor_sc = i, ov, scores[i]
    if mejor_i is None:
        return None, 0, 0.0
    return episodios[mejor_i], mejor_ov, float(mejor_sc)


print("memoria lista -- episodios guardados:", len(cargar()))
print()
print("Que entra a la memoria y que no:")
for m in ["los domingos juego al futbol", "te acordas de mi trabajo?",
          "que hora es?", "hola", "mi perro se llama Rocco"]:
    print(f"  {'GUARDA ' if vale_la_pena(m) else 'descarta'}  {m!r}")

## 4. Cuanto cuesta sacar las respuestas de Lora

Guardar solo los mensajes del usuario deja **menos tokens** por episodio, asi que
van a enganchar menos recuerdos. Esta celda mide cuanto, y sobre todo mira la otra
columna: **si el recuerdo recuperado es un dato cierto del usuario o un invento
del modelo**.

Un overlap mas alto no sirve de nada si el recuerdo que trae es algo que Lora se
invento.

In [ ]:
# Turnos reales: lo que dijo el usuario y lo que contesto Lora.
TURNOS_DEMO = [
    ("me gusta mucho el futbol, juego los domingos",
     "El futbol es lindo, yo miraria todos los partidos."),
    ("tengo un perro que se llama Rocco",
     "Rocco suena a un perro con caracter."),
    ("trabajo programando en python",
     "Programar en Python esta bueno, es prolijo."),
    ("mi hermano vive en Arequipa",
     "Arequipa es linda. Lo visitas seguido?"),
    ("estudio ingenieria en la universidad",
     "Ingenieria tiene lo suyo. Que ciclo vas?"),
]

# Dos corpus con los MISMOS turnos, cambiando solo que texto se indexa.
EPS_MIOS = [{"usuario": u, "importancia": _importancia(u)} for u, _ in TURNOS_DEMO]
EPS_CON_LORA = [{"usuario": f"{u} {r}", "importancia": _importancia(u)}
                for u, r in TURNOS_DEMO]

print("Tokens por episodio:\n")
for (u, r), a, b in zip(TURNOS_DEMO, EPS_MIOS, EPS_CON_LORA):
    t_mio = _tokenizar(a["usuario"])
    t_lora = [x for x in _tokenizar(b["usuario"]) if x not in set(t_mio)]
    print(f"  {u[:40]:<40}")
    print(f"    mios        : {t_mio}")
    print(f"    + de Lora   : {t_lora}   <- esto es lo que se descarta")
print()

In [ ]:
CONSULTAS_EVAL = [
    "viste el partido de futbol ayer?",      # partido != partidos
    "como esta Rocco?",
    "programas en python todavia?",          # programas != programando
    "donde vive tu hermano?",
    "que estudias?",                         # estudias != estudio
    "que opinas del clima?",                 # nada que ver: debe fallar
]

UMBRAL = memoria_episodica._OVERLAP_MINIMO   # 2, igual que el modulo real
n_mios = n_lora = 0

print(f"{'consulta':<34} {'solo mios':>11} {'con Lora':>10}")
print("-" * 58)
for q in CONSULTAS_EVAL:
    _, ov_m, _ = buscar(q, EPS_MIOS)
    _, ov_l, _ = buscar(q, EPS_CON_LORA)
    n_mios += ov_m >= UMBRAL
    n_lora += ov_l >= UMBRAL
    print(f"{q:<34} {ov_m:>7} {'OK' if ov_m >= UMBRAL else '--'}"
          f"  {ov_l:>6} {'OK' if ov_l >= UMBRAL else '--'}")
print("-" * 58)
print(f"pasan el umbral (>={UMBRAL}):   solo mios {n_mios}/{len(CONSULTAS_EVAL)}"
      f"   con Lora {n_lora}/{len(CONSULTAS_EVAL)}")

print("""
COMO LEER ESTO
--------------
Si 'con Lora' engancha mas, NO es porque recuerde mejor: es porque tiene mas
palabras para matchear, y esas palabras extra las escribio el modelo, no el
usuario. En el episodio de Arequipa los tokens extra son 'linda', 'visitas',
'seguido' -- ninguno es un dato del usuario.

Ese overlap de mas es ruido con forma de recall. Guardar solo los mensajes del
usuario engancha menos pero nunca devuelve como "recuerdo" algo que el 0.5B se
invento.

Si el recall queda demasiado bajo, lo que se ajusta es el UMBRAL (seccion 5),
no se vuelve a meter el texto de Lora.""")

## 5. Threshold: la decision del agente

La parte que hoy no existe. `buscar_relevante()` devuelve `None` y nadie decide
nada; aca el agente elige entre tres caminos:

| Decision | Cuando | Que hace |
|---|---|---|
| `USAR_MEMORIA` | overlap >= umbral | inyecta lo que conto el usuario y responde |
| `PREGUNTAR` | apela a la memoria y no hay episodio | responde y pide el dato que falta |
| `CONVERSAR` | el turno cuenta algo nuevo | conversa e invita a seguir contando |

Sin LLM: es una decision de control, mismo criterio que `agent_router` y
`agent_corrector`.

`UMBRAL_OVERLAP` arranca en 2 (el valor de `memoria_episodica.py`) pero ese valor
fue calibrado indexando tambien el texto de Lora. Con menos tokens por episodio
puede convenir bajarlo a 1 -- la celda de abajo lo barre.

In [ ]:
UMBRAL_OVERLAP = 2

# Marcas de que el usuario APELA a la memoria ("te acordas de X?"). Si pregunta
# por algo que deberia recordar y no hay episodio, el hueco es real.
#
# Ojo con la amplitud: la primera version incluia "mi " y rompia los turnos que
# CUENTAN algo ("ayer fui al cine con mi primo" caia en PREGUNTAR, cuando es un
# episodio nuevo). Van solo formas verbales de apelacion explicita.
_APELA_MEMORIA = (
    "te acordas", "te acuerdas", "recordas", "recuerdas", "te conte",
    "te conté", "te dije", "hablamos de", "sabes cual", "sabes que es mi",
)


def decidir(mensaje_usuario, episodio, overlap, umbral=None):
    """(decision, motivo). Sin LLM.

    El orden importa: USAR_MEMORIA gana sobre todo lo demas, y PREGUNTAR solo
    aplica cuando el usuario apela a la memoria Y no hay nada que la cubra.
    """
    umbral = UMBRAL_OVERLAP if umbral is None else umbral
    if episodio is not None and overlap >= umbral:
        return "USAR_MEMORIA", f"overlap {overlap} >= {umbral}"
    t = mensaje_usuario.lower()
    if any(p in t for p in _APELA_MEMORIA):
        return "PREGUNTAR", "apela a la memoria y no hay episodio que lo cubra"
    if vale_la_pena(mensaje_usuario):
        return "CONVERSAR", "cuenta algo nuevo: episodio a guardar"
    return "CONVERSAR", "turno trivial"


CASOS_DECISION = CONSULTAS_EVAL + [
    "me gusta el asado",              # CONVERSAR: cuenta algo
    "hola",                           # CONVERSAR: trivial
    "te acordas de mi perro?",        # apela, pero hay episodio de Rocco
    "te acordas de mi bicicleta?",    # apela y NO hay episodio -> PREGUNTAR
    "ayer fui al cine con mi primo",  # CONVERSAR (el bug de "mi ")
]

print(f"{'mensaje':<40} {'ov':>3}  decision")
print("-" * 74)
for q in CASOS_DECISION:
    ep, ov, _ = buscar(q, EPS_MIOS)
    d, motivo = decidir(q, ep, ov)
    print(f"{q:<40} {ov:>3}  {d:<13} {motivo}")

print("\nBarrido del umbral sobre CONSULTAS_EVAL:")
for u in (1, 2, 3):
    usa = []
    for q in CONSULTAS_EVAL:
        ep, ov, _ = buscar(q, EPS_MIOS)
        if decidir(q, ep, ov, umbral=u)[0] == "USAR_MEMORIA":
            usa.append(q)
    print(f"  umbral {u}: USAR_MEMORIA en {len(usa)}/{len(CONSULTAS_EVAL)}  {usa}")

print("""
Ojo al bajar el umbral: 'que opinas del clima?' NO tiene episodio que lo cubra.
Si aparece en la lista con umbral 1, es un falso positivo -- el bug que mato al
RAG viejo (score > 0 enganchaba cualquier cosa).""")

## 6. El agente completo

Las piezas juntas. Dos notas:

- **El recuerdo inyectado dice `(Antes me contaste: ...)`**, no
  `(Ya hablamos de esto antes: ...)`. El texto importa: lo que vuelve es lo que
  conto el usuario, y la frase tiene que decir eso.
- **La repregunta no se le pide al LLM**, es una plantilla. El fine-tune de v4 fue
  entrenado para *no* hacer preguntas propias (el system prompt del proyecto dice
  "no hagas preguntas propias ni cierres preguntando"), asi que pedirselo pelea
  contra su entrenamiento. Una frase fija es mas confiable y cuesta 0 ms.

In [ ]:
_REPREGUNTAS = [
    "No me acuerdo de eso. ¿Me lo contás?",
    "Eso no me lo contaste todavía. ¿Cómo fue?",
    "No lo tengo guardado. ¿Qué pasó?",
]
_INVITACIONES = [
    "¿Y qué más hiciste?",
    "Contame más de eso.",
    "¿Y cómo te fue?",
]


def agente_responder(mensaje_usuario, episodios=None, guardar_turno=False,
                     debug=True):
    """El agente de free chat. Devuelve la respuesta y el rastro de la decision."""
    eps = cargar() if episodios is None else episodios
    traza = {"mensaje": mensaje_usuario}

    # 1. recuperar (solo entre MIS mensajes) y 2. decidir
    ep, overlap, score = buscar(mensaje_usuario, eps)
    decision, motivo = decidir(mensaje_usuario, ep, overlap)
    recuerdo = ep["usuario"] if (ep and decision == "USAR_MEMORIA") else None
    traza.update({"decision": decision, "motivo": motivo, "overlap": overlap,
                  "score": round(score, 2), "recuerdo": recuerdo})

    # 3. PREGUNTAR no llama al LLM.
    #
    # El usuario pidio algo que el agente NO tiene, asi que no hay nada que
    # generar: si se le pregunta igual, el 0.5B contesta como si lo supiera
    # ("Si, me la acorde") y la repregunta queda contradiciendolo en la misma
    # frase. Medido en la primera version de este notebook:
    #   "te acordas de mi bicicleta?"
    #   -> "Si, me la acorde. [...] No me acuerdo de eso. Me lo contas?"
    # Admitir el hueco y pedir el dato es la respuesta correcta, y cuesta 0 ms.
    if decision == "PREGUNTAR":
        traza["respuesta"] = random.choice(_REPREGUNTAS)
        traza["seg"] = 0.0
        traza["llamo_llm"] = False
    else:
        contenido = (f"(Antes me contaste: {recuerdo})\n{mensaje_usuario}"
                     if recuerdo else mensaje_usuario)
        traza["prompt"] = contenido
        t0 = time.perf_counter()
        try:
            texto = llama_client.generar_respuesta(
                [{"role": "user", "content": contenido}]).strip()
        except Exception as e:
            texto = "Perdon, no te escuche bien."
            traza["error_llm"] = f"{type(e).__name__}: {e}"
        traza["seg"] = round(time.perf_counter() - t0, 2)
        traza["llamo_llm"] = True
        # Invitar a seguir contando, solo cuando el turno aporto un episodio.
        if decision == "CONVERSAR" and vale_la_pena(mensaje_usuario):
            texto = f"{texto} {random.choice(_INVITACIONES)}"
        traza["respuesta"] = texto

    # 4. guardar MI mensaje (nunca la respuesta de Lora). Sin LLM: 0 ms.
    if guardar_turno:
        nuevo = guardar(mensaje_usuario)
        traza["guardado"] = nuevo["usuario"] if nuevo else None

    if debug:
        print(f"> {mensaje_usuario}")
        print(f"  decision : {decision} ({motivo})")
        if recuerdo:
            print(f"  recuerdo : {recuerdo}")
        print(f"  respuesta: {traza['respuesta']}")
        print(f"  tiempo   : {traza['seg']}s"
              f"{'' if traza['llamo_llm'] else '  (sin LLM)'}")
        if traza.get("guardado"):
            print(f"  guardado : {traza['guardado']}")
        print()
    return traza


# Las tres ramas
for q in ["me gusta el futbol, juego los domingos",  # USAR_MEMORIA
          "te acordas de mi bicicleta?",             # PREGUNTAR (sin LLM)
          "ayer fui al cine con mi primo"]:          # CONVERSAR
    agente_responder(q, episodios=EPS_MIOS)

## 7. Conversacion acumulando memoria

Varios turnos seguidos, guardando de verdad: se ve como la memoria crece con
**solo mis mensajes** y empieza a engancharse sola.

In [ ]:
MIS_EPISODIOS.write_text("", encoding="utf-8")   # arrancar limpio

CONVERSACION = [
    "me llamo Adrian y estudio ingenieria",
    "tengo un perro que se llama Rocco",
    "los domingos juego al futbol con mis amigos",
    "mi perro Rocco se escapo el otro dia",   # deberia engancharse con el 2
    "el futbol del domingo lo gane",          # deberia engancharse con el 3
    "te acordas de mi trabajo?",              # hueco real: nunca lo conto
]

for mensaje in CONVERSACION:
    agente_responder(mensaje, guardar_turno=True)

print("=" * 62)
print("Memoria acumulada (solo mis mensajes):")
for e in cargar():
    print(f"  [{e['importancia']}] {e['usuario']}")

## 8. Si esto se lleva a `agent_free_chat.py`

Interfaz propuesta, en el estilo de los otros tres agentes (funciones de modulo,
sin clases, sin ROS 2 -- el orchestrator llama directo):

```python
# lora_brain/agent_free_chat.py
vale_la_pena(mensaje)                -> bool
guardar(mensaje)                     -> dict | None      # solo el mensaje del usuario
buscar(mensaje)                      -> (ep, overlap, score)
decidir(mensaje, ep, overlap)        -> (decision, motivo)
responder(mensaje)                   -> (texto, traza)
```

**Cero llamadas LLM extra por turno**: el agente solo llama al modelo para
conversar, igual que hoy. Memorizar, buscar y decidir son todas operaciones
locales.

En `orchestrator_node.py`, `_manejar_chat_libre()` cambia `self.responder(...)`
por `agent_free_chat.responder(...)`.

**Lo que falta definir antes de escribir el modulo:**

- **El umbral.** El 2 de `memoria_episodica.py` se calibro indexando tambien el
  texto de Lora. Con menos tokens por episodio puede convenir 1, pero hay que
  mirar los falsos positivos (seccion 5).
- **Que pasa con `memoria_episodica.py`.** Este agente lo reemplaza: son dos
  implementaciones de lo mismo con criterios distintos sobre que guardar. O se
  cambia ese modulo para que indexe solo `usuario`, o el agente lo deja sin uso.
- **`registro_chat.py` se queda como esta.** Guarda el turno completo a proposito:
  `curar.py` necesita ver la respuesta de Lora para armar datasets de fine-tuning.
  Son dos archivos con dos propositos distintos, no un duplicado.
- **Poda.** `memoria_episodica.py` corta en 300 (`_MAXIMO_EPISODIOS`). Con
  `importancia` se puede podar por prioridad en vez de por antiguedad.
- **Si la repregunta se dice en voz alta.** Hoy el aviso de salida de trivia queda
  solo en el log; la repregunta solo sirve si se escucha.

## Limpieza

In [ ]:
if MIS_EPISODIOS.exists():
    MIS_EPISODIOS.unlink()
    print(f"borrado {MIS_EPISODIOS}")
else:
    print("nada que borrar")